# Bab 15 - Deep Learning on Unseen Data: Introducing Federated Learning

**Nama:** Nanda Pratama ITM  
**NIM:** 101032330212  
**Kelas:** TK 47 01


> Notebook ini merupakan reproduksi edukatif dan pendalaman teori berdasarkan *Grokking Deep Learning* (Andrew W. Trask, Manning, 2019). Kode inti diadaptasi dari buku dan repositori pendamping resmi; perubahan kompatibilitas dijelaskan dalam dokumentasi.


## Capaian pembelajaran

Setelah menyelesaikan notebook ini, pembaca diharapkan mampu menjelaskan konsep utama bab, menghubungkannya dengan operasi NumPy, menjalankan contoh secara berurutan, dan menyebutkan batas interpretasinya.


## Masalah privasi

Mengumpulkan semua data mentah di server meningkatkan risiko kebocoran dan dapat melanggar batas penggunaan. Federated learning membawa proses training ke perangkat atau organisasi pemilik data lalu mengirim pembaruan model.


## Federated averaging

Setiap client memulai dari model global, melakukan beberapa pembaruan lokal, lalu server merata-ratakan parameter atau gradien dengan bobot sesuai jumlah data. Data yang tidak identik antarclient dapat membuat konvergensi lebih sulit.


## Ancaman terhadap pembaruan

Tidak mengirim data mentah belum menjamin privasi. Gradien dapat membocorkan informasi, client jahat dapat meracuni model, dan server dapat mengamati pembaruan individual.


## Secure aggregation

Secure aggregation membuat server hanya melihat jumlah pembaruan. Mask acak antarclient dapat dirancang saling membatalkan. Protokol nyata harus menangani client yang gagal, autentikasi, dan serangan kolusi.


## Homomorphic encryption

Enkripsi homomorfik memungkinkan operasi tertentu pada ciphertext, tetapi menambah biaya komputasi dan komunikasi. Privasi formal sering memerlukan kombinasi secure aggregation, differential privacy, dan tata kelola yang tepat.


## Reproduksi kode dan eksperimen

Jalankan sel dari atas ke bawah. `MODE_CEPAT=True` menjaga waktu eksekusi singkat; ubah ke `False` untuk konfigurasi yang lebih panjang.


In [1]:
from pathlib import Path
import gzip
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)
DATA_DIR = (Path.cwd().parent / "data").resolve()
MODE_CEPAT = True  # ubah ke False untuk eksperimen lebih panjang
assert DATA_DIR.is_dir(), f"Folder data tidak ditemukan: {DATA_DIR}"
print("Data:", DATA_DIR)
print("Mode cepat:", MODE_CEPAT)

def sigmoid(x):
    x = np.clip(x, -60, 60)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


Data: D:\Kuliah\DL\UPLOAD_GITHUB\Grokking-Deep-Learning\data
Mode cepat: True


In [2]:
from collections import Counter

with gzip.open(DATA_DIR / "spam.txt.gz", "rt", encoding="latin-1") as stream:
    spam = stream.read().splitlines()[:(1200 if MODE_CEPAT else 4000)]
with gzip.open(DATA_DIR / "ham.txt.gz", "rt", encoding="latin-1") as stream:
    ham = stream.read().splitlines()[:(1200 if MODE_CEPAT else 4000)]
messages = spam + ham
targets = np.array([1.0] * len(spam) + [0.0] * len(ham))
counts = Counter(word for msg in messages for word in set(msg.lower().split()))
vocab = [word for word, _ in counts.most_common(800)]
word2index = {word: i for i, word in enumerate(vocab)}

def vectorize(items):
    x = np.zeros((len(items), len(vocab)))
    for row, msg in enumerate(items):
        for word in set(msg.lower().split()):
            if word in word2index: x[row, word2index[word]] = 1
    return x

x = vectorize(messages)
order = np.random.default_rng(15).permutation(len(x))
x, targets = x[order], targets[order]
print(x.shape, targets.mean())


(2400, 800) 0.5


In [3]:
# Empat client melakukan update lokal, server menjalankan federated averaging
clients = np.array_split(np.arange(len(x)), 4)
global_w = np.zeros(x.shape[1])
alpha = 0.2
for round_id in range(8 if MODE_CEPAT else 25):
    local_weights = []
    for ids in clients:
        w = global_w.copy()
        pred = sigmoid(x[ids] @ w)
        grad = x[ids].T @ (pred - targets[ids]) / len(ids)
        w -= alpha * grad
        local_weights.append(w)
    global_w = np.average(local_weights, axis=0, weights=[len(ids) for ids in clients])
prediction = sigmoid(x @ global_w) >= 0.5
print("federated training accuracy:", np.mean(prediction == targets))


federated training accuracy: 0.9533333333333334


In [4]:
# Mask berpasangan saling membatalkan pada secure aggregation sederhana
rng = np.random.default_rng(150)
updates = [w - global_w for w in local_weights]
masks = [rng.normal(size=global_w.shape) for _ in range(len(updates)-1)]
masked = []
for i, update in enumerate(updates):
    add = masks[i] if i < len(masks) else 0
    subtract = masks[i-1] if i > 0 else 0
    masked.append(update + add - subtract)
assert np.allclose(np.sum(masked, axis=0), np.sum(updates, axis=0))
print("mask cancel:", np.max(np.abs(np.sum(masked, axis=0)-np.sum(updates, axis=0))))


mask cancel: 8.881784197001252e-16


## Interpretasi dan kesimpulan

Hasil harus dibaca bersama asumsi, konfigurasi, seed, serta data yang digunakan. Keberhasilan menjalankan kode tidak membuktikan generalisasi, kausalitas, keamanan, atau kelayakan deployment. Bandingkan hasil dengan teori di atas dan jelaskan setiap perbedaan sebelum mengubah konfigurasi.
